In [ ]:
%pip install scikit-learn matplotlib pillow kagglehub tensorflow

In [1]:
import numpy as np
import tensorflow as tf
from pathlib import Path
import random
import shutil


In [2]:


Path.cwd()

# Для определения рабочей папки директории

WindowsPath('c:/Users/user/Desktop/detects_ns')

In [3]:


root = Path("C:/Users/user/Desktop/detects_ns/"
            "archive/dataset_2/main_dataset_2")

#root = Path("<здесь код из ячейки выше>"
#            "archive/dataset_2/main_dataset_2")


extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
n = 2  # фото в каждой папке

for folder in sorted([root, *[p for p in root.rglob("*") if p.is_dir()]]):
    level = len(folder.relative_to(root).parts)
    indent = "  " * level
    print(f"{indent}📁 {folder.name or root.name}")

    photos = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    )
    for i, photo in enumerate(photos[:n], start=1):
        print(f"{indent} 🖼️ {photo.name}")

📁 main_dataset_2
  📁 synthetic_defect
   🖼️ synth_defect_1.png
   🖼️ synth_defect_10.png
  📁 synthetic_non_defect
   🖼️ synth_non_defect_1.png
   🖼️ synth_non_defect_10.png


In [4]:


defect_dir = Path(
    "C:/Users/user/Desktop/detects_ns/"
    "archive/dataset_2/main_dataset_2/"
    "synthetic_defect"
)

non_defect_dir = Path(
    "C:/Users/user/Desktop/detects_ns/"
    "archive/dataset_2/main_dataset_2/"
    "synthetic_non_defect"
)

# Папка результата создаётся в текущей рабочей директории.
output_dir = Path("nut_dataset")

random.seed(42)
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

files = []

for folder, label in [
    (defect_dir, "defect"),
    (non_defect_dir, "non_defect"),
]:
    if not folder.is_dir():
        raise FileNotFoundError(f"Папка не найдена: {folder}")

    # Ищет изображения и в самой папке, и во всех вложенных папках.
    files.extend(
        (path, label)
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )

random.shuffle(files)

# Создаём подпапки классов.
for label in ("defect", "non_defect"):
    (output_dir / label).mkdir(parents=True, exist_ok=True)

# Копируем все найденные изображения.
for index, (source, label) in enumerate(files):
    destination = output_dir / label / f"{index:06d}_{source.name}"
    shutil.copy2(source, destination)

print(f"Скопировано изображений: {len(files)}")
print(f"С дефектом: {sum(label == 'defect' for _, label in files)}")
print(f"Без дефекта: {sum(label == 'non_defect' for _, label in files)}")
print(f"Результат: {output_dir.resolve()}")

Скопировано изображений: 260
С дефектом: 130
Без дефекта: 130
Результат: C:\Users\user\Desktop\detects_ns\nut_dataset


In [5]:


data_dir = Path("nut_dataset")  # при необходимости укажите полный путь
img_size = (180, 180)
batch_size = 32
seed = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary",
)

class_names = train_ds.class_names
print("Порядок классов:", class_names)

autotune = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(autotune)
val_ds = val_ds.cache().prefetch(autotune)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*img_size, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(train_ds, validation_data=val_ds, epochs=15)
model.save("nut_defect_model.keras")

Found 260 files belonging to 2 classes.
Using 208 files for training.
Found 260 files belonging to 2 classes.
Using 52 files for validation.
Порядок классов: ['defect', 'non_defect']
Epoch 1/15


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 220ms/step - accuracy: 0.4856 - loss: 1.0312 - val_accuracy: 0.6154 - val_loss: 0.6909
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 192ms/step - accuracy: 0.5144 - loss: 0.6961 - val_accuracy: 0.4038 - val_loss: 0.7054
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 191ms/step - accuracy: 0.5433 - loss: 0.6907 - val_accuracy: 0.4038 - val_loss: 0.6962
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 196ms/step - accuracy: 0.5817 - loss: 0.6611 - val_accuracy: 0.4231 - val_loss: 0.7157
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 194ms/step - accuracy: 0.6971 - loss: 0.5955 - val_accuracy: 0.7115 - val_loss: 0.5838
Epoch 6/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 183ms/step - accuracy: 0.7212 - loss: 0.5352 - val_accuracy: 0.6731 - val_loss: 0.5760
Epoch 7/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 192ms/step - accuracy: 0.8317 - loss: 0.4592 - val_accuracy: 0.7115 - val_loss: 0.5395
Epoch 8/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 200ms/step - accuracy: 0.9038 - loss: 0.3171 - val_accuracy: 0.8269 - val_loss: 0.3985
Epo

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

# Сохранить модель целиком
model.save("nut_defect_model.keras")
print("Модель сохранена: nut_defect_model.keras")

In [6]:
loss, accuracy = model.evaluate(val_ds)
print(f"Точность на валидации: {accuracy:.1%}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.7500 - loss: 0.5015
Точность на валидации: 75.0%


In [ ]:


#image_path = r"c:/Users/user/Desktop/detects_ns/test.png"
image_path = r"c:/Users/user/Desktop/detects_ns/test_2.jpg"

img = tf.keras.utils.load_img(image_path, target_size=(180, 180))
x = tf.keras.utils.img_to_array(img)
x = tf.expand_dims(x, axis=0)  # добавляем размерность batch

probability = float(model.predict(x, verbose=0)[0][0])

class_names = ["defect", "non_defect"]
predicted_class = class_names[int(probability >= 0.5)]

print("Предсказание:", predicted_class)
print(f"Вероятность non_defect: {probability:.1%}")

Предсказание: defect
Вероятность non_defect: 5.3%
